In [10]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import json

In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
from src.mlp_markov import *
from src.markov.evals import *

In [13]:
df_all = pd.read_csv('data/pbp/pbp_2005_2025_plays.csv')

In [14]:
train_df = df_all[df_all['season'] <= 2020].reset_index(drop = True)
test_df = df_all[df_all['season'] >= 2021].reset_index(drop = True)

In [15]:
train_df.T

,0,1,2,3,4,5,6,7,8,9,...,603825,603826,603827,603828,603829,603830,603831,603832,603833,603834
season,2005,2005,2005,2005,2005,2005,2005,2005,2005,2005,...,2020,2020,2020,2020,2020,2020,2020,2020,2020,2020
season_type,REG,REG,REG,REG,REG,REG,REG,REG,REG,REG,...,REG,REG,REG,REG,REG,REG,REG,REG,REG,REG
week,1,1,1,1,1,1,1,1,1,1,...,17,17,17,17,17,17,17,17,17,17
game_id,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,2005_01_ARI_NYG,...,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI,2020_17_WAS_PHI
game_date,2005-09-11,2005-09-11,2005-09-11,2005-09-11,2005-09-11,2005-09-11,2005-09-11,2005-09-11,2005-09-11,2005-09-11,...,2021-01-03,2021-01-03,2021-01-03,2021-01-03,2021-01-03,2021-01-03,2021-01-03,2021-01-03,2021-01-03,2021-01-03
desc,(14:53) 13-K.Warner pass incomplete to 83-E.Ed...,(14:49) 28-J.Arrington right guard to ARI 36 f...,(14:22) 13-K.Warner pass incomplete to 80-B.Jo...,(14:08) 13-K.Warner pass to 11-L.Fitzgerald to...,(13:41) 13-K.Warner pass incomplete to 11-L.Fi...,"(13:28) (Shotgun) PENALTY on ARI-75-L.Davis, F...",(13:28) (Shotgun) 28-J.Arrington right guard t...,"(13:06) 10-S.Player punts 53 yards to NYG 14, ...",(12:48) 10-E.Manning pass to 17-P.Burress to N...,(12:10) 21-T.Barber right guard to NYG 28 for ...,...,(1:55) 24-A.Gibson right end to PHI 39 for 1 y...,(1:49) 24-A.Gibson right end to PHI 36 for 3 y...,"(1:02) 5-T.Way punts 28 yards to PHI 8, Center...",(:56) (Shotgun) 7-N.Sudfeld pass short right t...,(:50) (Shotgun) 7-N.Sudfeld pass incomplete de...,(:44) (Shotgun) 7-N.Sudfeld pass incomplete sh...,(:38) (Shotgun) 7-N.Sudfeld scrambles up the m...,"(:16) (No Huddle, Shotgun) 7-N.Sudfeld pass in...",(:11) (Shotgun) 7-N.Sudfeld pass incomplete sh...,(:06) (Shotgun) 7-N.Sudfeld pass short middle ...
drive_number,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,...,24.0,24.0,24.0,25.0,25.0,25.0,25.0,25.0,25.0,25.0
play_id,66.0,85.0,106.0,136.0,157.0,176.0,199.0,220.0,245.0,266.0,...,3717.0,3755.0,3793.0,3812.0,3846.0,3868.0,3890.0,3911.0,3933.0,3955.0
play_number,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,1.0,2.0,...,8.0,9.0,10.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0
posteam,ARI,ARI,ARI,ARI,ARI,ARI,ARI,ARI,NYG,NYG,...,WAS,WAS,WAS,PHI,PHI,PHI,PHI,PHI,PHI,PHI


In [2]:
import numpy as np
import pandas as pd
import time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, mean_absolute_error
from src import util
plays_df = pd.read_csv('data/pbp/pbp_2005_2025_plays.csv')
drives_df = pd.read_csv('data/pbp/pbp_2005_2025_drives.csv')

In [ ]:
# ============================================================
# Covariate-dependent Markov, bin-based features (state_id-consistent)
#   baseline    : state (down, ytg_bin, field_bin)
#   +time       : state + game_seconds_remaining
#   +time+score : state + time + score_differential
# Play-level + Drive-level simulation (greedy & sample).
# Features derived from state_id so simulation reconstruction is exact.
# ============================================================

SPLIT_MODE = 'cut2020' # pick 4 train/test splits - 'cut2020', 'all', 'bundle', 'yearly'
MIN_NEXT_COUNT = 300 # exclude rare 'next_states' 
FIRST_DOWN_STATES = list(range(1, 61))
ABS_IDS = list(util.ABSORBING_STATES_MAP.keys())
ABS_POINTS = {241: 7, 242: 3, 243: 0, 244: 0, 245: 0, 246: 0} # for calculating drive MAE
MAX_SIM_PLAYS = 40
N_SIM_DRIVES = 3000            # only used if sampling is enabled below
RNG = np.random.RandomState(42)

train_list, test_list, test_drives_list = util.get_train_test_sets(
    plays_df, drives_df, split_mode=SPLIT_MODE, train_frac=0.75)

_, ydstogo_labels = util.get_ydstogo_cuts_labels()
_, field_labels = util.get_field_position_cuts_labels()
state_values = util.get_state_values(
    ["down", "binned_ydstogo_id", "binned_field_position_id"],
    [[1, 2, 3, 4], ydstogo_labels, field_labels])
_, id_to_state, _ = util.generate_state_ids(state_values, util.ABSORBING_STATES)

N_YTG = len(ydstogo_labels)
N_FIELD = len(field_labels)

def make_features(state_ids, score_diff, gsr, spread, total, rain, snow, wind, use_time, use_score, use_betting, use_weather):
    downs = np.array([id_to_state[s][0] for s in state_ids])
    ytgs = np.array([id_to_state[s][1] for s in state_ids])
    fields = np.array([id_to_state[s][2] for s in state_ids])
    cols = []
    for dn in [1, 2, 3, 4]:
        cols.append((downs == dn).astype(float))
    for y in range(1, N_YTG + 1):
        cols.append((ytgs == y).astype(float))
    for f in range(1, N_FIELD + 1):
        cols.append((fields == f).astype(float))
    X = np.column_stack(cols)
    if use_time:
        X = np.column_stack([X, np.asarray(gsr, float)])
    if use_score:
        X = np.column_stack([X, np.asarray(score_diff, float)])
    if use_betting:
        X = np.column_stack([X, np.asarray(spread, float)])
        X = np.column_stack([X, np.asarray(total, float)])
    if use_weather:
        X = np.column_stack([X, np.asarray(rain, float)])
        X = np.column_stack([X, np.asarray(snow, float)])
        X = np.column_stack([X, np.asarray(wind, float)])
    return X

MODELS = {
    'baseline':    dict(use_time=False, use_score=False, use_betting = False, use_weather = False),
    '+time':       dict(use_time=True,  use_score=False, use_betting = False, use_weather = False),
    '+time+score': dict(use_time=True,  use_score=True, use_betting = False, use_weather = False),
    '+time+score+betting': dict(use_time = True, use_score = True, use_betting = True, use_weather = False),
    '+time+score+betting+weather': dict(use_time = True, use_score = True, use_betting = True, use_weather = True)
}

REQ = ['cur_state_id', 'next_state_id', 'score_differential', 'game_seconds_remaining', 'spread_line', 'total_line', 'rain', 'snow', 'wind']

def top_k_acc(proba, y_true, classes, k):
    topk = np.argsort(proba, axis=1)[:, -k:]
    ca = np.array(classes)
    return np.mean([y_true[i] in ca[topk[i]] for i in range(len(y_true))])

def eval_fold(train_df, test_df, test_drives_df):
    tr = train_df.dropna(subset=REQ).copy()
    te = test_df.dropna(subset=REQ).copy()
    tr = tr[tr['cur_state_id'].isin(id_to_state.keys())]
    te = te[te['cur_state_id'].isin(id_to_state.keys())]
    vc = tr['next_state_id'].value_counts()
    keep = set(vc[vc >= MIN_NEXT_COUNT].index)
    tr = tr[tr['next_state_id'].isin(keep)]
    te = te[te['next_state_id'].isin(keep)]
    print(f"    train={len(tr):,} test={len(te):,} classes={len(keep)}", flush=True)

    y_tr = tr['next_state_id'].values
    y_te = te['next_state_id'].values
    tr_states = tr['cur_state_id'].values.astype(int)
    te_states = te['cur_state_id'].values.astype(int)
    tr_sd, tr_gsr = tr['score_differential'].values, tr['game_seconds_remaining'].values
    tr_spread, tr_total = tr['spread_line'].values, tr['total_line'].values
    tr_rain, tr_snow, tr_wind = tr['rain'].values, tr['snow'].values, tr['wind'].values
    te_sd, te_gsr = te['score_differential'].values, te['game_seconds_remaining'].values
    te_spread, te_total = te['spread_line'].values, te['total_line'].values
    te_rain, te_snow, te_wind = te['rain'].values, te['snow'].values, te['wind'].values

    dd = test_drives_df.dropna(subset=['start_state_id', 'drive_result_id']).copy()
    dd = dd[dd['start_state_id'].isin([s for s in id_to_state if s <= 240])]

    fold_res = {}
    for name, cfg in MODELS.items():
        t0 = time.time()
        print(f"    fitting {name} ...", flush=True)
        X_tr = make_features(tr_states, tr_sd, tr_gsr, tr_spread, tr_total, tr_rain, tr_snow, tr_wind, **cfg)
        scaler = StandardScaler().fit(X_tr)
        X_tr = scaler.transform(X_tr)
        clf = MLPClassifier(hidden_layer_sizes=(10,), max_iter=1500, random_state=7)
        # clf = LogisticRegression(max_iter=1000, solver='saga', tol=1e-3)
        clf.fit(X_tr, y_tr)
        classes = clf.classes_

        # ---- play-level ----
        X_te = scaler.transform(make_features(te_states, te_sd, te_gsr, te_spread, te_total, te_rain, te_snow, te_wind, **cfg))
        proba = clf.predict_proba(X_te)
        ll = log_loss(y_te, proba, labels=classes)
        t5 = top_k_acc(proba, y_te, classes, 5)
        t10 = top_k_acc(proba, y_te, classes, 10)
        fd_cols = [i for i, c in enumerate(classes) if c in FIRST_DOWN_STATES]
        fd_prob = proba[:, fd_cols].sum(axis=1)
        fd_true = np.isin(y_te, FIRST_DOWN_STATES).astype(int)
        fd_auc = roc_auc_score(fd_true, fd_prob) if 0 < fd_true.sum() < len(fd_true) else np.nan

        # ---- drive-level simulation ----
        abs_set = set(ABS_IDS)

        # ===== toggle: FULL vs SAMPLED =====
        sample_drives = dd                                              # FULL (all drives)
        # sample_drives = dd.sample(n=min(N_SIM_DRIVES, len(dd)), random_state=1)  # SAMPLED

        print(f"    simulating {len(sample_drives):,} drives ({name}) ...", flush=True)
        s0_arr = sample_drives['start_state_id'].values.astype(int)
        sd_arr = sample_drives['start_score_differential'].fillna(0).values
        gsr_arr = sample_drives['start_time_rem'].fillna(1800).values
        spread_arr = sample_drives['spread_line'].fillna(0).values
        total_arr = sample_drives['total_line'].fillna(0).values
        rain_arr = sample_drives['rain'].fillna(0).values
        snow_arr = sample_drives['snow'].fillna(0).values
        wind_arr = sample_drives['wind'].fillna(0).values
        actual_arr = sample_drives['drive_result_id'].values.astype(int)

        def simulate(s0, sd, gsr, spread, total, rain, snow, wind, mode):
            state = s0
            for _ in range(MAX_SIM_PLAYS):
                if state in abs_set:
                    return state
                x = scaler.transform(make_features([state], [sd], [gsr], [spread], [total], [rain], [snow], [wind], **cfg))
                p = clf.predict_proba(x)[0]
                state = int(classes[np.argmax(p)]) if mode == 'greedy' \
                        else int(RNG.choice(classes, p=p))
            return state if state in abs_set else 246

        for mode in ['greedy', 'sample']:
            preds = np.empty(len(s0_arr), dtype=int)
            for i in range(len(s0_arr)):
                preds[i] = simulate(s0_arr[i], sd_arr[i], gsr_arr[i], spread_arr[i], total_arr[i], rain_arr[i], snow_arr[i], wind_arr[i], mode)
                if (i + 1) % 5000 == 0:
                    print(f"      {name} {mode}: {i+1:,}/{len(s0_arr):,}", flush=True)
            fold_res.setdefault(name, {})
            fold_res[name][f'drive_final_acc_{mode}'] = accuracy_score(actual_arr, preds)
            fold_res[name][f'drive_score_acc_{mode}'] = accuracy_score(
                np.isin(actual_arr, [241, 242]), np.isin(preds, [241, 242]))
            fold_res[name][f'drive_td_acc_{mode}'] = accuracy_score(
                actual_arr == 241, preds == 241)
            fold_res[name][f'drive_points_mae_{mode}'] = mean_absolute_error(
                [ABS_POINTS.get(a, 0) for a in actual_arr],
                [ABS_POINTS.get(p, 0) for p in preds])

        fold_res[name].update({'state_log_loss': ll, 'top5_acc': t5,
                               'top10_acc': t10, 'first_down_roc_auc': fd_auc})
        print(f"    {name} done in {time.time()-t0:.1f}s (ll={ll:.3f})", flush=True)
    return fold_res

print(f"split_mode={SPLIT_MODE}, folds={len(train_list)}", flush=True)
all_folds = []
for k, (a, b, c) in enumerate(zip(train_list, test_list, test_drives_list)):
    print(f"  fold {k+1}/{len(train_list)}", flush=True)
    all_folds.append(eval_fold(a, b, c))

models = list(MODELS.keys())
mets = sorted({m for f in all_folds for md in f.values() for m in md})
summary = pd.DataFrame(
    {mdl: {mt: np.nanmean([f[mdl][mt] for f in all_folds if mt in f[mdl]])
           for mt in mets} for mdl in models}).T

play_cols = ['state_log_loss', 'top5_acc', 'top10_acc', 'first_down_roc_auc']
drive_cols = [c for c in mets if c.startswith('drive_')]

print("\n" + "=" * 75)
print(f"PLAY-LEVEL (split={SPLIT_MODE})")
print("=" * 75)
print(summary[play_cols].round(4).to_string())
print("\n" + "=" * 75)
print("DRIVE-LEVEL (simulation)")
print("=" * 75)
print(summary[drive_cols].round(4).to_string())


Getting train/test sets using split mode "cut2020" with training fraction of games = 0.75
split_mode=cut2020, folds=1
  fold 1/1
    train=441,533 test=111,387 classes=196
    fitting baseline ...
    simulating 29,126 drives (baseline) ...
      baseline greedy: 5,000/29,126
      baseline greedy: 10,000/29,126
      baseline greedy: 15,000/29,126
      baseline greedy: 20,000/29,126
      baseline greedy: 25,000/29,126
      baseline sample: 5,000/29,126
      baseline sample: 10,000/29,126
      baseline sample: 15,000/29,126
      baseline sample: 20,000/29,126
      baseline sample: 25,000/29,126
    baseline done in 167.3s (ll=2.353)
    fitting +time ...
    simulating 29,126 drives (+time) ...
      +time greedy: 5,000/29,126
      +time greedy: 10,000/29,126
      +time greedy: 15,000/29,126
      +time greedy: 20,000/29,126
      +time greedy: 25,000/29,126
      +time sample: 5,000/29,126
      +time sample: 10,000/29,126
      +time sample: 15,000/29,126
      +time sample:

In [22]:
summary[[x for x in summary.columns if 'greedy' in x]]

,drive_final_acc_greedy,drive_points_mae_greedy,drive_score_acc_greedy,drive_td_acc_greedy
baseline,0.374339,1.926423,0.649969,0.780917
+time,0.373618,1.929170,0.649969,0.780231
+time+score,0.376090,1.928277,0.649076,0.781123
+time+score+betting,0.371970,1.938371,0.645986,0.780917
+time+score+betting+weather,0.372657,1.937822,0.645986,0.781055


In [23]:
summary[[x for x in summary.columns if 'sample' in x]]

,drive_final_acc_sample,drive_points_mae_sample,drive_score_acc_sample,drive_td_acc_sample
baseline,0.259116,2.501751,0.574744,0.693504
+time,0.259631,2.586830,0.572890,0.673625
+time+score,0.257124,2.640424,0.565920,0.665454
+time+score+betting,0.261691,2.542230,0.575397,0.682895
+time+score+betting+weather,0.257674,2.552599,0.574092,0.681281


In [20]:
summary[['state_log_loss', 'top5_acc', 'top10_acc', 'first_down_roc_auc']]

,state_log_loss,top5_acc,top10_acc,first_down_roc_auc
baseline,2.352771,0.703700,0.862668,0.706226
+time,2.342587,0.706591,0.863225,0.707022
+time+score,2.341686,0.706752,0.863225,0.705159
+time+score+betting,2.340574,0.705558,0.864311,0.708255
+time+score+betting+weather,2.339224,0.704957,0.864356,0.708199


In [ ]:
summary[['state_log_loss', 'top5_acc', 'top10_acc', 'first_down_roc_auc']]

In [ ]:
#add in before we create play level dataset add in rain column and snow columns

In [6]:
# ============================================================
# Covariate-dependent Markov: NESTED covariates, WHOLE structure
#
# Rows (10): baseline_A (counting), baseline_B (no cov),
#            +score, +time, +home, +spread, +total, +rain, +snow, +wind
# Play-level (7): top1/3/5/10 acc, true/predicted first-down rate, first-down AUC
# Drive-level (8): score acc/actual/pred, td acc/actual/pred, final-state acc, points MAE
# predicted_first_down_rate = mean of P(next in first-down states) over test plays
# ============================================================
import numpy as np
import pandas as pd
import time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, mean_absolute_error
from src import util
from src.markov import simple_markov
import caffeine

caffeine.on(display = False)


# ---------------- CONFIG ----------------
SPLIT_MODE     = 'cut2020'
MIN_NEXT_COUNT = 300
FIRST_DOWN_STATES = list(range(1, 61))
ABS_IDS    = list(util.ABSORBING_STATES_MAP.keys())
ABS_POINTS = {241: 7, 242: 3, 243: 0, 244: 0, 245: 0, 246: 0}
SCORE_IDS  = [241, 242]
TD_ID      = 241
MAX_SIM_PLAYS = 40
N_SIM_DRIVES  = 3000
SIM_MODE    = 'sample'
FULL_DRIVES = True
N_SIM_REPEATS = 1
RNG = np.random.RandomState(42)

NESTED_ORDER = ['score_differential', 'game_seconds_remaining', 'is_home',
                'spread_line', 'total_line', 'rain', 'snow', 'wind']
NESTED_LABEL = {'score_differential': '+score', 'game_seconds_remaining': '+time',
                'is_home': '+home', 'spread_line': '+spread', 'total_line': '+total',
                'rain': '+rain', 'snow': '+snow', 'wind': '+wind'}
BOOL_COVARS = {'rain', 'snow'}

DRIVE_COL = {'score_differential': 'start_score_differential',
             'game_seconds_remaining': 'start_time_rem',
             'spread_line': 'spread_line', 'total_line': 'total_line',
             'rain': 'rain', 'snow': 'snow', 'wind': 'wind'}
DRIVE_DEFAULT = {'score_differential': 0.0, 'game_seconds_remaining': 1800.0,
                 'spread_line': 0.0, 'total_line': 44.0,
                 'rain': 0.0, 'snow': 0.0, 'wind': 0.0}

# ---------------- SPLIT ----------------
train_list, test_list, test_drives_list = util.get_train_test_sets(
    plays_df, drives_df, split_mode=SPLIT_MODE, train_frac=0.75)
print(f"split_mode={SPLIT_MODE}, folds={len(train_list)}", flush=True)

# ---------------- state id <-> (down, ytg, field) ----------------
_, ydstogo_labels = util.get_ydstogo_cuts_labels()
_, field_labels   = util.get_field_position_cuts_labels()
state_values = util.get_state_values(
    ["down", "binned_ydstogo_id", "binned_field_position_id"],
    [[1, 2, 3, 4], ydstogo_labels, field_labels])
_, id_to_state, _ = util.generate_state_ids(state_values, util.ABSORBING_STATES)
N_YTG, N_FIELD = len(ydstogo_labels), len(field_labels)

def state_onehot(state_ids):
    downs = np.array([id_to_state[s][0] for s in state_ids])
    ytgs  = np.array([id_to_state[s][1] for s in state_ids])
    flds  = np.array([id_to_state[s][2] for s in state_ids])
    cols  = [(downs == d).astype(float) for d in [1, 2, 3, 4]]
    cols += [(ytgs == y).astype(float) for y in range(1, N_YTG + 1)]
    cols += [(flds == f).astype(float) for f in range(1, N_FIELD + 1)]
    return np.column_stack(cols)

def covar_matrix(df, covars):
    if not covars:
        return np.zeros((len(df), 0))
    cols = []
    for c in covars:
        if c == 'is_home':
            cols.append((df['posteam_type'] == 'home').astype(float).to_numpy())
        elif c in BOOL_COVARS:
            cols.append(df[c].astype(bool).astype(float).to_numpy())
        else:
            cols.append(df[c].to_numpy(float))
    return np.column_stack(cols)

def covar_vec_drive(row, covars):
    if not covars:
        return np.zeros(0)
    vals = []
    for c in covars:
        if c == 'is_home':
            vals.append(1.0 if row.get('posteam_type') == 'home' else 0.0)
        else:
            v = row.get(DRIVE_COL[c], np.nan)
            if pd.isna(v):
                vals.append(DRIVE_DEFAULT[c])
            elif c in BOOL_COVARS:
                vals.append(1.0 if bool(v) else 0.0)
            else:
                vals.append(float(v))
    return np.array(vals, float)

# ---------------- metrics ----------------
def top_k_acc(proba, y_true, classes, k):
    topk = np.argsort(proba, axis=1)[:, -k:]
    ca = np.array(classes)
    return np.mean([y_true[i] in ca[topk[i]] for i in range(len(y_true))])

def play_metrics(proba_full, y_te, all_classes):
    fd_cols = [i for i, c in enumerate(all_classes) if c in FIRST_DOWN_STATES]
    fd_prob = proba_full[:, fd_cols].sum(axis=1)
    fd_true = np.isin(y_te, FIRST_DOWN_STATES).astype(int)
    return {
        'top_1_accuracy':  top_k_acc(proba_full, y_te, all_classes, 1),
        'top_3_accuracy':  top_k_acc(proba_full, y_te, all_classes, 3),
        'top_5_accuracy':  top_k_acc(proba_full, y_te, all_classes, 5),
        'top_10_accuracy': top_k_acc(proba_full, y_te, all_classes, 10),
        'true_first_down_rate':      fd_true.mean(),
        'predicted_first_down_rate': fd_prob.mean(),
        'first_down_roc_auc': roc_auc_score(fd_true, fd_prob)
                              if 0 < fd_true.sum() < len(fd_true) else np.nan,
    }

def drive_metrics(actual, preds):
    a_score = np.isin(actual, SCORE_IDS); p_score = np.isin(preds, SCORE_IDS)
    a_td = actual == TD_ID;               p_td = preds == TD_ID
    return {
        'drive_score_accuracy':       accuracy_score(a_score, p_score),
        'drive_score_actual_rate':    a_score.mean(),
        'drive_score_predicted_rate': p_score.mean(),
        'drive_td_accuracy':          accuracy_score(a_td, p_td),
        'drive_td_actual_rate':       a_td.mean(),
        'drive_td_predicted_rate':    p_td.mean(),
        'drive_final_state_accuracy': accuracy_score(actual, preds),
        'drive_points_mae': mean_absolute_error(
            [ABS_POINTS.get(a, 0) for a in actual], [ABS_POINTS.get(p, 0) for p in preds]),
    }

def average_dicts(ds):
    return {k: float(np.mean([d[k] for d in ds])) for k in ds[0]}

# ---------------- per-fold evaluation ----------------
def eval_fold(train_df, test_df, test_drives_df):
    base_req = ['cur_state_id', 'next_state_id', 'posteam_type',
                'score_differential', 'game_seconds_remaining',
                'spread_line', 'total_line', 'rain', 'snow', 'wind']
    tr = train_df.dropna(subset=base_req).copy()
    te = test_df.dropna(subset=base_req).copy()
    tr = tr[tr['cur_state_id'].isin(id_to_state.keys())]
    te = te[te['cur_state_id'].isin(id_to_state.keys())]
    vc = tr['next_state_id'].value_counts()
    keep = set(vc[vc >= MIN_NEXT_COUNT].index)
    tr = tr[tr['next_state_id'].isin(keep)]
    te = te[te['next_state_id'].isin(keep)]
    print(f"    train={len(tr):,} test={len(te):,} classes={len(keep)}", flush=True)

    dd = test_drives_df.dropna(subset=['start_state_id', 'drive_result_id']).copy()
    dd = dd[dd['start_state_id'].isin([s for s in id_to_state if s <= 240])]
    sim_drives = dd if FULL_DRIVES else dd.sample(n=min(N_SIM_DRIVES, len(dd)), random_state=1)
    s0_arr = sim_drives['start_state_id'].values.astype(int)
    actual = sim_drives['drive_result_id'].values.astype(int)
    cov_rows = sim_drives.to_dict('records')
    print(f"    drives to simulate: {len(s0_arr):,}", flush=True)

    te_states = te['cur_state_id'].values.astype(int)
    y_te = te['next_state_id'].values
    all_classes = sorted(keep)
    cls_idx = {c: i for i, c in enumerate(all_classes)}
    abs_set = set(ABS_IDS)
    Xs_te = state_onehot(te_states)
    Xs_tr = state_onehot(tr['cur_state_id'].values.astype(int))
    y_tr = tr['next_state_id'].values

    results = {}

    # ===== BASELINE A: counting =====
    t0 = time.time()
    print(f"    === baseline_A (counting) ===", flush=True)
    P = simple_markov.fit_markov_chain(tr)
    P_idx = {s: i for i, s in enumerate(P.index)}
    P_np = P.to_numpy(); P_cols = np.array(P.columns)
    proba_full = np.zeros((len(te_states), len(all_classes)))
    for sid in np.unique(te_states):
        if sid not in P_idx: continue
        rows = np.where(te_states == sid)[0]
        prow = P_np[P_idx[sid]]
        for j, c in enumerate(P_cols):
            if c in cls_idx: proba_full[rows, cls_idx[c]] = prow[j]
    proba_full = np.where(proba_full.sum(axis=1, keepdims=True) > 0,
                          proba_full, 1.0 / len(all_classes))
    proba_full /= proba_full.sum(axis=1, keepdims=True)
    res = play_metrics(proba_full, y_te, all_classes)
    dms = []
    for rep in range(N_SIM_REPEATS):
        preds = np.empty(len(s0_arr), dtype=int)
        for i in range(len(s0_arr)):
            state = s0_arr[i]
            for _ in range(MAX_SIM_PLAYS):
                if state in abs_set or state not in P_idx: break
                prow = P_np[P_idx[state]]
                if prow.sum() <= 0: break
                state = int(P_cols[np.argmax(prow)]) if SIM_MODE == 'greedy' \
                        else int(RNG.choice(P_cols, p=prow / prow.sum()))
            preds[i] = state if state in abs_set else 246
        dms.append(drive_metrics(actual, preds))
    res.update(average_dicts(dms))
    results['baseline_A_counting'] = res
    print(f"    done in {time.time()-t0:.1f}s (top1={res['top_1_accuracy']:.3f}, "
          f"ptsMAE={res['drive_points_mae']:.3f})", flush=True)

    # ===== multinomial WHOLE =====
    def run_whole(label, covars):
        t0 = time.time()
        print(f"    === {label} | covars={covars} ===", flush=True)
        X_tr = np.column_stack([Xs_tr, covar_matrix(tr, covars)])
        scaler = StandardScaler().fit(X_tr)
        clf = MLPClassifier(hidden_layer_sizes=(10,), max_iter=1500, random_state=7)
        # clf = LogisticRegression(max_iter=1000, solver='saga', tol=1e-3)
        clf.fit(scaler.transform(X_tr), y_tr)
        classes = clf.classes_

        X_te = np.column_stack([Xs_te, covar_matrix(te, covars)])
        Pm = clf.predict_proba(scaler.transform(X_te))
        proba_full = np.zeros((len(te_states), len(all_classes)))
        for j, c in enumerate(classes):
            if c in cls_idx: proba_full[:, cls_idx[c]] = Pm[:, j]
        proba_full /= proba_full.sum(axis=1, keepdims=True).clip(min=1e-12)
        res = play_metrics(proba_full, y_te, all_classes)

        dms = []
        for rep in range(N_SIM_REPEATS):
            preds = np.empty(len(s0_arr), dtype=int)
            for i in range(len(s0_arr)):
                state = s0_arr[i]; cov_vec = covar_vec_drive(cov_rows[i], covars)
                for _ in range(MAX_SIM_PLAYS):
                    if state in abs_set: break
                    X = np.column_stack([state_onehot([state]), cov_vec[None, :]])
                    p = clf.predict_proba(scaler.transform(X))[0]
                    state = int(classes[np.argmax(p)]) if SIM_MODE == 'greedy' \
                            else int(RNG.choice(classes, p=p))
                preds[i] = state if state in abs_set else 246
                if (i + 1) % 5000 == 0:
                    print(f"          rep{rep+1} {i+1:,}/{len(s0_arr):,}", flush=True)
            dms.append(drive_metrics(actual, preds))
        res.update(average_dicts(dms))
        print(f"    done in {time.time()-t0:.1f}s (top1={res['top_1_accuracy']:.3f}, "
              f"ptsMAE={res['drive_points_mae']:.3f})", flush=True)
        return res

    results['baseline_B_nocov'] = run_whole('baseline_B', [])
    covars = []
    for c in NESTED_ORDER:
        covars = covars + [c]
        results[NESTED_LABEL[c]] = run_whole(NESTED_LABEL[c], list(covars))
    return results

# ---------------- run ----------------
all_folds = []
for k, (a, b, c) in enumerate(zip(train_list, test_list, test_drives_list)):
    print(f"  fold {k+1}/{len(train_list)}", flush=True)
    all_folds.append(eval_fold(a, b, c))

# ---------------- report: two tables ----------------
ROWS = ['baseline_A_counting', 'baseline_B_nocov'] + [NESTED_LABEL[c] for c in NESTED_ORDER]
PLAY_COLS = ['top_1_accuracy', 'top_3_accuracy', 'top_5_accuracy', 'top_10_accuracy',
             'true_first_down_rate', 'predicted_first_down_rate', 'first_down_roc_auc']
DRIVE_COLS = ['drive_score_accuracy', 'drive_score_actual_rate', 'drive_score_predicted_rate',
              'drive_td_accuracy', 'drive_td_actual_rate', 'drive_td_predicted_rate',
              'drive_final_state_accuracy', 'drive_points_mae']

summary = pd.DataFrame(
    {m: {mt: np.nanmean([f[m][mt] for f in all_folds]) for mt in PLAY_COLS + DRIVE_COLS}
     for m in ROWS}).T

pd.set_option('display.width', 220)
print("\n" + "=" * 100)
print(f"PLAY-LEVEL (10 x 7)   split={SPLIT_MODE}, sim={SIM_MODE}, repeats={N_SIM_REPEATS}")
print("=" * 100)
print(summary[PLAY_COLS].round(4).to_string())
print("\n" + "=" * 100)
print(f"DRIVE-LEVEL (10 x 8)")
print("=" * 100)
print(summary[DRIVE_COLS].round(4).to_string())

import os
os.makedirs('plots', exist_ok=True)
summary[PLAY_COLS].round(4).to_csv('plots/nested_play_level.csv')
summary[DRIVE_COLS].round(4).to_csv('plots/nested_drive_level.csv')
print("\nsaved: plots/nested_play_level.csv, plots/nested_drive_level.csv")
caffeine.off()

Getting train/test sets using split mode "cut2020" with training fraction of games = 0.75
split_mode=cut2020, folds=1
  fold 1/1
    train=441,533 test=111,387 classes=196
    drives to simulate: 29,126
    === baseline_A (counting) ===
Fitting Markov chain
    done in 2.0s (top1=0.328, ptsMAE=2.562)
    === baseline_B | covars=[] ===
          rep1 5,000/29,126
          rep1 10,000/29,126
          rep1 15,000/29,126
          rep1 20,000/29,126
          rep1 25,000/29,126
    done in 157.3s (top1=0.324, ptsMAE=2.513)
    === +score | covars=['score_differential'] ===
          rep1 5,000/29,126
          rep1 10,000/29,126
          rep1 15,000/29,126
          rep1 20,000/29,126
          rep1 25,000/29,126
    done in 188.3s (top1=0.324, ptsMAE=2.601)
    === +time | covars=['score_differential', 'game_seconds_remaining'] ===
          rep1 5,000/29,126
          rep1 10,000/29,126
          rep1 15,000/29,126
          rep1 20,000/29,126
          rep1 25,000/29,126
    done in 

In [8]:
summary[[x for x in summary.columns if 'drive' in x]]

,drive_score_accuracy,drive_score_actual_rate,drive_score_predicted_rate,drive_td_accuracy,drive_td_actual_rate,drive_td_predicted_rate,drive_final_state_accuracy,drive_points_mae
baseline_A_counting,0.570350,0.375163,0.330873,0.681728,0.220662,0.194843,0.255373,2.562041
baseline_B_nocov,0.573886,0.375163,0.324109,0.691341,0.220662,0.173556,0.255373,2.512978
+score,0.568118,0.375163,0.319165,0.673591,0.220662,0.202568,0.254755,2.601284
+time,0.576255,0.375163,0.316453,0.681831,0.220662,0.194259,0.258326,2.543913
+home,0.574504,0.375163,0.326718,0.679462,0.220662,0.196972,0.261210,2.558642
+spread,0.570418,0.375163,0.345705,0.673865,0.220662,0.209366,0.253004,2.593284
+total,0.573131,0.375163,0.321912,0.677264,0.220662,0.196697,0.255682,2.571551
+rain,0.572272,0.375163,0.329156,0.683788,0.220662,0.188388,0.256918,2.548033
+snow,0.573886,0.375163,0.334478,0.683341,0.220662,0.190551,0.259665,2.544977
+wind,0.569045,0.375163,0.330049,0.685607,0.220662,0.187255,0.254515,2.550436
